# Taller de Machine Learning supervisado - Vocales

**Asignatura:** Inteligencia Artificial - Universidad del Magdalena

**Curso:** IA-2026-II  |  **Tema:** Machine Learning - Reconocimiento óptico de caracteres (OCR)

**Integrantes:** Carlos Pinto, León Cantillo, Santiago Gonzales, Jorge García.

## Contenido del equipo
- Clases asignadas: vocales **A, E, I, O, U**.
- La vocal **O** se representa con la carpeta **`0`** (no existe un directorio propio para la letra O, por similitud visual con el dígito cero).

## Alcance de este archivo
Este notebook analiza **únicamente las vocales**. Se excluyen los números del 1 al 9 y las letras que no son vocales. Incluye la descripción del problema, la inspección del conjunto de datos, el preprocesamiento (conversión a blanco y negro), el diseño de experimentos y la implementación de **árboles de decisión** con `GridSearchCV`, junto con el análisis de resultados y las conclusiones.


## 1. Descripción del problema y objetivo

El reconocimiento óptico de caracteres (OCR) convierte imágenes de texto manuscrito o impreso en texto digital editable. En este trabajo se aborda una versión reducida del problema: clasificar una imagen de un carácter manuscrito en una de las cinco vocales del español (**A, E, I, O, U**). Es un problema de **clasificación supervisada multiclase**, porque cada imagen tiene asociada una etiqueta conocida durante el entrenamiento.

El conjunto de datos asignado contiene también dígitos y consonantes. Para este trabajo se conservan solo las carpetas de las vocales y se usa la carpeta `0` como aproximación de la vocal **O**. Esa decisión debe considerarse al interpretar los resultados, porque la letra O y el dígito 0 son símbolos distintos aunque compartan una forma cerrada.

| Elemento | Descripción |
|---|---|
| Casos | 5.070 imágenes de vocales |
| Variables de entrada | 7.500 píxeles por imagen (75 de ancho x 100 de alto), inicialmente en escala de grises |
| Variable objetivo | Vocal: `A`, `E`, `I`, `O`, `U` |
| Número de clases | 5 |
| Valores faltantes | 0 en las etiquetas; todas las imágenes son legibles |
| Origen | Kaggle: *Handwriting Letter and Number Dataset* |

**Objetivo:** construir y evaluar un clasificador basado en **árboles de decisión** que reconozca las cinco vocales a partir de imágenes binarizadas, ajustando sus hiperparámetros con `GridSearchCV` y reportando métricas de desempeño que permitan decidir si el modelo es adecuado.

La documentación del repositorio y la guía del trabajo se usaron para definir el problema, las variables y las etapas de experimentación.


## 2. Configuración y carga del dataset

Se configura la ruta del dataset y se cargan solo las carpetas de clase correspondientes a vocales. El dataset completo (**CNN letter Dataset**) no se incluye en este repositorio por su peso. Las imágenes binarizadas se guardan en `dataset/blanco y negro/`, una carpeta derivada que tampoco se versiona.


In [ ]:
import os
import re
import time
import zipfile
import hashlib
import random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from dotenv import load_dotenv
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             classification_report, confusion_matrix)

random.seed(42)
np.random.seed(42)
plt.rcParams["figure.dpi"] = 100



# ---------------------------------------------------------------
# CONFIGURACIÓN CENTRALIZADA
# ---------------------------------------------------------------

def encontrar_raiz_proyecto():
    """Busca la raíz del proyecto localizando el archivo .env."""
    actual = Path.cwd().resolve()

    for carpeta in [actual, *actual.parents]:
        if (carpeta / ".env").is_file():
            return carpeta

    raise FileNotFoundError(
        "No se encontró el archivo .env. "
        "Debe estar en la raíz del proyecto."
    )


RAIZ_PROYECTO = encontrar_raiz_proyecto()
RUTA_ENV = RAIZ_PROYECTO / ".env"

load_dotenv(RUTA_ENV, override=False)


def obtener_ruta(nombre_variable):
    """Obtiene una ruta del .env y la resuelve desde la raíz del proyecto."""
    valor = os.getenv(nombre_variable)

    if not valor:
        raise ValueError(
            f"No se definió {nombre_variable} en el archivo .env."
        )

    ruta = Path(valor).expanduser()

    if not ruta.is_absolute():
        ruta = RAIZ_PROYECTO / ruta

    return ruta.resolve()


# Rutas principales
RUTA_DATASET = obtener_ruta("DATASET_ORIGINAL")
RUTA_BN = obtener_ruta("DATASET_BINARIZADO")

# ---------------------------------------------------------------
# VALIDACIONES
# ---------------------------------------------------------------

EXT = {".png", ".jpg", ".jpeg", ".bmp", ".gif", ".tif", ".tiff"}

VOCALES = ["A", "E", "I", "O", "U"]


def tiene_clases(carpeta):
    """Comprueba que existan subcarpetas con imágenes."""
    if not carpeta.is_dir():
        return False

    subcarpetas = [
        d for d in carpeta.iterdir()
        if d.is_dir() and not d.name.startswith(("__", "."))
    ]

    clases_con_imagenes = [
        d for d in subcarpetas
        if any(
            f.is_file() and f.suffix.lower() in EXT
            for f in d.iterdir()
        )
    ]

    return len(clases_con_imagenes) >= 2


if not tiene_clases(RUTA_DATASET):
    raise FileNotFoundError(
        f"No se encontró un dataset válido en:\n{RUTA_DATASET}\n"
        "Comprueba DATASET_ORIGINAL en el archivo .env."
    )

if RUTA_DATASET == RUTA_BN:
    raise ValueError(
        "El dataset original y el binarizado no pueden compartir la misma ruta."
    )

print("Configuración cargada correctamente.")
print("Raíz del proyecto:", RAIZ_PROYECTO)
print("Dataset original:", RUTA_DATASET)
print("Dataset binarizado:", RUTA_BN)


## 3. Organización de archivos y carpetas

Se muestra el árbol de carpetas restringido a las clases de vocales. Cada subcarpeta es una clase.


In [ ]:
for carpeta, subs, archivos in os.walk(RAIZ_PROYECTO):
    if "__MACOSX" in carpeta:
        continue
    nombre = Path(carpeta).name
    # Solo mostrar carpetas de clase vocales o la raíz
    if carpeta == str(RAIZ_PROYECTO) or nombre in VOCALES or nombre == "0":
        nivel = len(Path(carpeta).relative_to(RAIZ_PROYECTO).parts)
        print("  " * nivel + f"{nombre}/  ({len(archivos)} archivos)")

print("\nEjemplos de nombres de archivo por vocal:")
for d in sorted(x for x in RAIZ_PROYECTO.iterdir() if x.is_dir() and (x.name in VOCALES or x.name == "0")):
    print(f"  {d.name}: {[f.name for f in sorted(d.iterdir())[:3]]}")


## 4. Cantidad de imágenes por clase

Se construye una tabla con la ruta, clase y formato de cada imagen. La carpeta `0` se mapea a la vocal `O`. Se excluyen del análisis las carpetas de números del 1 al 9 y demás letras que no sean vocales.


In [ ]:
registros = []
for p in RAIZ_PROYECTO.rglob("*"):
    if p.is_file() and p.suffix.lower() in EXT and "__MACOSX" not in p.parts:
        clase = p.parent.name
        # Mapear carpeta '0' a vocal 'O'
        if clase == "0":
            clase = "O"
        # Filtrar solo vocales (excluir números 1-9 y otras letras)
        if clase in VOCALES:
            registros.append({"ruta": p, "clase": clase, "formato": p.suffix.lower()})

df = pd.DataFrame(registros)
CLASES = sorted(df["clase"].unique())

print("Total de imágenes (vocales):", len(df))
print("Clases (vocales):", CLASES)

conteo = df["clase"].value_counts().reindex(VOCALES).fillna(0).astype(int)
print("\nConteo por vocal:")
print(conteo.to_string())
print("\nTotal:", int(conteo.sum()))
print("Razón entre la clase más y menos frecuente:", round(conteo.max() / conteo.min(), 3))

fig, ax = plt.subplots(figsize=(7, 4))
barras = ax.bar(conteo.index, conteo.values, color="steelblue", edgecolor="black")
ax.set_title("Número de imágenes por vocal")
ax.set_xlabel("Clase (vocal)")
ax.set_ylabel("Cantidad de imágenes")
ax.set_ylim(0, conteo.max() * 1.1)
for b, v in zip(barras, conteo.values):
    ax.text(b.get_x() + b.get_width() / 2, v, str(v), ha="center", va="bottom")
plt.tight_layout()
plt.show()
df.head()


## 5. Muestras aleatorias por clase

Se seleccionan imágenes aleatorias de cada vocal y se muestran en una cuadrícula para observar variaciones de escritura, fondo, iluminación, grosor de trazo, rotación o ruido.


In [ ]:
N_POR_CLASE = 8
rng = np.random.RandomState(7)

fig, axes = plt.subplots(len(CLASES), N_POR_CLASE, figsize=(2 * N_POR_CLASE, 2.3 * len(CLASES)))
for i, c in enumerate(CLASES):
    sub = df[df["clase"] == c]
    idx = rng.choice(len(sub), size=min(N_POR_CLASE, len(sub)), replace=False)
    for j, k in enumerate(idx):
        with Image.open(sub.iloc[k]["ruta"]) as im:
            a = np.array(im)
        ax = axes[i, j]
        ax.imshow(a, cmap="gray", vmin=0, vmax=255)
        ax.set_xticks([]); ax.set_yticks([])
        if j == 0:
            ax.set_ylabel(f"Clase {c}", fontsize=12, rotation=0, labelpad=35, va="center")
fig.suptitle("Imágenes aleatorias de cada vocal", fontsize=14)
plt.tight_layout()
plt.show()

## 6. Dimensiones, canales y formato

Se verifican todas las imágenes del subconjunto de vocales: tamaño, modo de color, número de canales, tipo de dato y rango de intensidades. Se determina si todas tienen el mismo tamaño.


In [ ]:
filas = []
for ruta, clase in zip(df["ruta"], df["clase"]):
    try:
        with Image.open(ruta) as im:
            a = np.array(im)
            filas.append({
                "clase": clase,
                "ancho": im.size[0],
                "alto": im.size[1],
                "modo": im.mode,
                "canales": 1 if a.ndim == 2 else a.shape[2],
                "dtype": str(a.dtype),
                "min": int(a.min()),
                "max": int(a.max()),
                "media": float(a.mean()),
                "desv": float(a.std()),
            })
    except Exception as e:
        filas.append({"clase": clase, "ancho": np.nan, "alto": np.nan, "modo": "ERROR",
                      "canales": np.nan, "dtype": "", "min": np.nan, "max": np.nan,
                      "media": np.nan, "desv": np.nan})

info = pd.DataFrame(filas)
info["ruta"] = df["ruta"].values

print("Tamaños (ancho, alto) y frecuencia:")
print(info.groupby(["ancho", "alto"]).size().to_string())
print("\nModos de color:\n", info["modo"].value_counts().to_string())
print("\nNúmero de canales:\n", info["canales"].value_counts().to_string())
print("\nTipo de dato:\n", info["dtype"].value_counts().to_string())
print("\nFormato de archivo:\n", df["formato"].value_counts().to_string())

mismo_tamano = info[["ancho", "alto"]].drop_duplicates().shape[0] == 1
print("\n¿Todas las imágenes tienen el mismo tamaño?", mismo_tamano)


## 7. Calidad de los datos

Se revisa: imágenes corruptas o ilegibles, duplicados exactos por contenido, imágenes idénticas en clases distintas, brillo/contraste atípicos y el patrón de los nombres de archivo.


In [ ]:
# 1) Imágenes corruptas o ilegibles
corruptas = []
for ruta in df["ruta"]:
    try:
        with Image.open(ruta) as im:
            im.verify()
    except Exception:
        corruptas.append(ruta)
print("Imágenes corruptas:", len(corruptas))

# 2) Duplicados exactos por contenido (hash MD5)
def md5(ruta):
    h = hashlib.md5()
    with open(ruta, "rb") as f:
        h.update(f.read())
    return h.hexdigest()

df["hash"] = [md5(r) for r in df["ruta"]]
print("Archivos duplicados exactos:", int(df["hash"].duplicated().sum()))
print("Contenidos repetidos en clases distintas:", int((df.groupby("hash")["clase"].nunique() > 1).sum()))

# 3) Valores faltantes en la tabla de metadatos
print("\nValores faltantes en la tabla de metadatos:")
print(df[["ruta", "clase", "formato"]].isna().sum().to_string())

# 4) Imágenes casi uniformes (poco contraste) o muy oscuras/claras
UMBRAL_DESV = 10
poco_contraste = info[info["desv"] < UMBRAL_DESV]
print(f"\nImágenes con desviación estándar < {UMBRAL_DESV} (poco contraste): {len(poco_contraste)}")
if len(poco_contraste):
    print(poco_contraste["clase"].value_counts().to_string())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for c in CLASES:
    axes[0].hist(info.loc[info["clase"] == c, "media"], bins=30, alpha=0.5, label=c)
axes[0].set_title("Brillo medio por imagen, según clase")
axes[0].set_xlabel("Intensidad media del píxel (0-255)")
axes[0].set_ylabel("Número de imágenes")
axes[0].legend(title="Clase")

info.boxplot(column="desv", by="clase", ax=axes[1])
axes[1].set_title("Contraste (desv. estándar) por clase")
axes[1].set_xlabel("Clase")
axes[1].set_ylabel("Desviación estándar de píxeles")
plt.suptitle("")
plt.tight_layout()
plt.show()

# 5) Patrón de los nombres de archivo
patron = re.compile(r"^aug(\d+)_(\d+)$")
nombres = df["ruta"].apply(lambda p: Path(p).stem)
coincide = nombres.apply(lambda s: bool(patron.match(s)))
print(f"\nArchivos que siguen el patrón 'aug<id>_<sufijo>': {coincide.sum()} de {len(df)}")


## 8. Exploración gráfica adicional

### Imagen promedio por clase y distribución de intensidades

La imagen promedio permite ver qué forma "típica" tiene cada vocal; el histograma de intensidades ayuda a decidir si es necesario normalizar los píxeles.


In [ ]:
# Imagen promedio por clase
if not mismo_tamano:
    print("Las imágenes no tienen todas el mismo tamaño; se omite la imagen promedio.")
else:
    H, W = int(info["alto"].iloc[0]), int(info["ancho"].iloc[0])
    promedios = {}
    for c in CLASES:
        suma = np.zeros((H, W), dtype=np.float64)
        rutas = df.loc[df["clase"] == c, "ruta"]
        for r in rutas:
            with Image.open(r) as im:
                suma += np.array(im.convert("L"), dtype=np.float64)
        promedios[c] = suma / len(rutas)

    fig, axes = plt.subplots(1, len(CLASES), figsize=(3 * len(CLASES), 4))
    for ax, c in zip(axes, CLASES):
        ax.imshow(promedios[c], cmap="gray", vmin=0, vmax=255)
        ax.set_title(f"Promedio clase {c}")
        ax.axis("off")
    plt.suptitle("Imagen promedio por clase", y=1.02)
    plt.tight_layout()
    plt.show()

# Histograma de intensidades de píxel (muestra de 300 imágenes por clase)
fig, ax = plt.subplots(figsize=(8, 4))
for c in CLASES:
    sub = df[df["clase"] == c].sample(min(300, (df["clase"] == c).sum()), random_state=42)
    pix = np.concatenate([np.array(Image.open(r).convert("L")).ravel() for r in sub["ruta"]])
    ax.hist(pix, bins=64, range=(0, 255), alpha=0.4, density=True, label=c)
ax.set_title("Distribución de intensidades de píxel por clase")
ax.set_xlabel("Intensidad (0 = negro, 255 = blanco)")
ax.set_ylabel("Densidad")
ax.legend(title="Clase")
plt.tight_layout()
plt.show()


## 9. Resumen cuantitativo de la inspección


In [ ]:
resumen_final = pd.DataFrame({
    "Aspecto": [
        "Total de imágenes",
        "Número de clases",
        "Clases",
        "Imagen más/menos frecuente (razón)",
        "Formatos de archivo",
        "Tamaños distintos (ancho x alto)",
        "Modos de color",
        "Canales",
        "Imágenes corruptas",
        "Duplicados exactos",
        "Contenido repetido entre clases",
        "Imágenes con poco contraste",
    ],
    "Valor": [
        len(df),
        len(CLASES),
        ", ".join(CLASES),
        round(conteo.max() / conteo.min(), 3),
        ", ".join(df["formato"].unique()),
        "; ".join(f"{int(a)}x{int(b)}" for a, b in info[["ancho", "alto"]].drop_duplicates().values),
        ", ".join(info["modo"].unique()),
        ", ".join(str(int(x)) for x in info["canales"].dropna().unique()),
        len(corruptas),
        int(df["hash"].duplicated().sum()),
        int((df.groupby("hash")["clase"].nunique() > 1).sum()),
        len(poco_contraste),
    ],
})
resumen_final


## 10. Tratamientos definidos a partir de la inspección

La inspección mostró que las 5.070 imágenes de vocales tienen el mismo tamaño (75x100), un solo canal, formato JPG, ninguna imagen corrupta, ningún duplicado exacto y clases prácticamente balanceadas (razón 1,02 entre la clase más y menos frecuente). Por tanto, no se eliminan registros ni se imputan valores.

Los tratamientos que sí se justifican son:

- **Conversión a blanco y negro:** el brillo y el contraste varían entre imágenes, así que se aplica un umbral de Otsu por imagen para separar tinta y fondo. Esto reduce el ruido de iluminación.
- **Mapeo de la clase `0` a `O`:** mantiene las cinco vocales y conserva la convención del dataset.
- **Vectorización:** cada imagen binarizada se aplana en 7.500 variables binarias (un píxel por variable), sin escalado, porque los árboles de decisión no lo requieren.
- **Partición agrupada:** los nombres `aug<id>_<sufijo>` indican variantes aumentadas del mismo trazo base. Para evitar fuga de información, la partición de entrenamiento y prueba se hace por grupo base, no por imagen individual.
- **Balanceo:** no es necesario; las clases están equilibradas.


## 11. Preprocesamiento: conversión a blanco y negro

### 11.1 Binarización con el método de Otsu

La inspección evidenció que la intensidad del fondo y del trazo cambia de una imagen a otra. Un umbral fijo produciría máscaras distintas según la iluminación, por lo que se usa el **método de Otsu**: para cada imagen se elige el umbral que maximiza la varianza entre dos grupos de píxeles (tinta y fondo). El resultado se guarda con tinta negra (`0`) y fondo blanco (`255`) en el directorio del dataset en blanco y negro.


In [ ]:
def umbral_otsu(imagen):
    """Calcula el umbral de Otsu de una imagen en escala de grises."""
    histograma = np.bincount(
        imagen.ravel(), minlength=256
    ).astype(np.float64)

    probabilidad = histograma / histograma.sum()
    omega = np.cumsum(probabilidad)
    media_acumulada = np.cumsum(
        probabilidad * np.arange(256)
    )
    media_total = media_acumulada[-1]

    with np.errstate(divide="ignore", invalid="ignore"):
        varianza_entre = (
            (media_total * omega - media_acumulada) ** 2
            / (omega * (1 - omega))
        )

    varianza_entre[~np.isfinite(varianza_entre)] = -1

    return int(np.argmax(varianza_entre))


def binarizar(ruta):
    """Devuelve la imagen binaria (tinta=0, fondo=255) y su umbral."""
    with Image.open(ruta) as im:
        gris = np.asarray(im.convert("L"))

    umbral = umbral_otsu(gris)
    binaria = np.where(gris > umbral, 255, 0).astype(np.uint8)

    return binaria, umbral

# ---------------------------------------------------------------
# GENERACIÓN DEL DATASET BINARIZADO
# ---------------------------------------------------------------

inicio = time.time()

RUTA_BN.mkdir(parents=True, exist_ok=True)
registros_bn = []
for ruta, clase in zip(df["ruta"], df["clase"]):
    binaria, umbral = binarizar(ruta)
    destino = RUTA_BN / clase / f"{ruta.stem}.png"
    destino.parent.mkdir(parents=True, exist_ok=True)
    Image.fromarray(binaria, mode="L").save(destino)
    registros_bn.append({
        "ruta_original": ruta,
        "ruta_bn": destino,
        "clase": clase,
        "umbral": umbral,
        "tinta": float((binaria == 0).mean()),
    })

df_bn = pd.DataFrame(registros_bn)
print(f"Imágenes convertidas: {len(df_bn)} en {time.time() - inicio:.1f} s")
print(df_bn.groupby("clase")["umbral"].agg(["mean", "min", "max"]).round(1).to_string())

In [ ]:
muestras = df_bn.groupby("clase", group_keys=False).sample(1, random_state=42)
fig, axes = plt.subplots(len(muestras), 2, figsize=(5.5, 2.2 * len(muestras)))
for ax, (_, fila) in zip(axes, muestras.iterrows()):
    with Image.open(fila["ruta_original"]) as im:
        original = np.asarray(im.convert("L"))
    binaria = np.asarray(Image.open(fila["ruta_bn"]))
    ax[0].imshow(original, cmap="gray", vmin=0, vmax=255)
    ax[0].set_title(f"Original ({fila['clase']})")
    ax[0].axis("off")
    ax[1].imshow(binaria, cmap="gray", vmin=0, vmax=255)
    ax[1].set_title(f"B/N, umbral = {fila['umbral']}")
    ax[1].axis("off")
plt.suptitle("Conversión a blanco y negro con umbral de Otsu", y=1.01)
plt.tight_layout()
plt.show()

fila = df_bn.sample(1, random_state=7).iloc[0]
with Image.open(fila["ruta_original"]) as im:
    gris = np.asarray(im.convert("L"))
fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(gris.ravel(), bins=64, range=(0, 255), color="gray")
ax.axvline(fila["umbral"], color="red", label=f"Umbral Otsu = {fila['umbral']}")
ax.set_title("Intensidades de una imagen y umbral de Otsu")
ax.set_xlabel("Intensidad (0 = negro, 255 = blanco)")
ax.set_ylabel("Número de píxeles")
ax.legend()
plt.tight_layout()
plt.show()

### 11.2 Matriz de imágenes binarizadas

Todas las imágenes binarizadas tienen el mismo tamaño (75×100) y modo `'L'`. Se compilan en una matriz tridimensional `X_img` de dimensiones $(N, 100, 75)$ de tipo `uint8`, con valores `0` (tinta) y `255` (fondo).

In [ ]:
ANCHO, ALTO = 75, 100

X_img = np.stack([
    np.asarray(Image.open(ruta_bn), dtype=np.uint8)
    for ruta_bn in df_bn["ruta_bn"]
])
y = df_bn["clase"].to_numpy()

print("Forma de X_img:", X_img.shape, "| dtype:", X_img.dtype)
print("Valores únicos:", np.unique(X_img))
print("Tamaño uniforme:", X_img.shape[1:] == (100, 75))

### 11.3 Comparación original vs. binarizada

Se verifica que la binarización con Otsu conserva la morfología del trazo y elimina el fondo.

In [ ]:
idx = np.random.RandomState(3).choice(len(df_bn), 5, replace=False)

fig, axes = plt.subplots(2, 5, figsize=(13, 6))
for j, k in enumerate(idx):
    with Image.open(df_bn.iloc[k]["ruta_original"]) as im:
        axes[0, j].imshow(im.convert("L"), cmap="gray", vmin=0, vmax=255)
    axes[0, j].set_title(f"Original ({df_bn.iloc[k]['clase']})", fontsize=10)
    axes[1, j].imshow(X_img[k], cmap="gray", vmin=0, vmax=255)
    axes[1, j].set_title("Binarizada", fontsize=10)
    axes[0, j].axis("off"); axes[1, j].axis("off")
plt.suptitle("Imágenes antes y después de la binarización")
plt.tight_layout()
plt.show()

### 11.4 Aplanado (Flattening): de matriz de píxeles a vector de características

Los algoritmos clásicos de Machine Learning supervisado requieren como entrada una matriz bidimensional $(N, D)$, donde cada fila representa una muestra y cada columna una característica numérica. Mediante el aplanado (*flattening*), cada imagen de $100 \times 75$ píxeles se convierte en un vector unidimensional de $7.500$ valores, preservando la correspondencia posicional de cada píxel de forma completamente reversible.

In [ ]:
# ---------- Flattening: de matriz de píxeles a vector ----------
# Cada imagen (100 x 75) se convierte en un vector de 7500 valores
X_flat = X_img.reshape(len(X_img), -1)

print("Forma antes:", X_img.shape)
print("Forma después:", X_flat.shape)
print("¿Reversible?", np.array_equal(X_flat[0].reshape(ALTO, ANCHO), X_img[0]))

# ---------- Visualización del flattening en una imagen ----------
k = None   # None = imagen aleatoria en cada ejecución; o un número (0 a 5069) para fijarla
T = 20     # tamaño (T x T) del recorte que se amplía

if k is None:
    k = int(np.random.randint(len(X_img)))

fila = df_bn.iloc[k]

print(
    f"\nImagen mostrada: índice {k} | "
    f"archivo {fila['ruta_original'].name} | "
    f"clase {fila['clase']}"
)

# Se busca la zona de T x T con más contraste (donde el trazo cruza el fondo)
f0, c0 = max(
    ((f, c) for f in range(0, ALTO - T + 1, 5) for c in range(0, ANCHO - T + 1, 5)),
    key=lambda p: X_img[k, p[0]:p[0]+T, p[1]:p[1]+T].std(),
)
recorte = X_img[k, f0:f0+T, c0:c0+T]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(X_img[k], cmap="gray", vmin=0, vmax=255)
axes[0].add_patch(plt.Rectangle((c0-0.5, f0-0.5), T, T, fill=False, edgecolor="red", linewidth=2))
axes[0].set_title(f"Imagen completa (clase {df_bn.loc[k, 'clase']})\nRecuadro rojo = zona ampliada")
axes[0].axis("off")

axes[1].imshow(recorte, cmap="gray", vmin=0, vmax=255)
axes[1].set_title(f"Recorte {T}x{T} en escala de grises")
axes[1].axis("off")

axes[2].imshow(recorte, cmap="gray", vmin=0, vmax=255)
for i in range(T):
    for j in range(T):
        v = int(recorte[i, j])
        axes[2].text(j, i, str(v), ha="center", va="center", fontsize=8,
                     color="white" if v < 128 else "black")
axes[2].set_title("El mismo recorte con el valor de cada píxel (0/255)")
axes[2].axis("off")
plt.tight_layout()
plt.show()

vec = recorte.reshape(-1)
print("Matriz del recorte:", recorte.shape, "-> vector aplanado:", vec.shape)
print("Primeros 20 valores del vector:", vec[:20])
print("Imagen completa:", X_img[k].shape, "-> vector:", X_flat[k].shape)

### 11.5 Normalización y escalado de variables $[0, 1]$

Siguiendo la metodología del cuaderno de referencia de la asignatura ([`PCA_MNIST_dimensionality_reduction.ipynb`](https://github.com/idanis/IA_Unimagdalena/blob/main/PCA/PCA_MNIST_dimensionality_reduction.ipynb)), los valores de intensidad de los píxeles binarizados $(0 = tinta, 255 = fondo)$ se escalan al intervalo $[0.0, 1.0]$ dividiendo por $255.0$.

Esta normalización es fundamental en visión computacional y aprendizaje de máquinas porque:
- Estandariza la escala para optimizadores basados en gradientes (Redes Neuronales y Regresión Logística).
- Facilita el cálculo numérico de covarianzas en técnicas lineales como PCA.
- Almacena la matriz en tipo de dato `float32`, manteniendo un consumo de memoria eficiente.


In [ ]:
# Normalización de los píxeles al rango [0.0, 1.0]
X = (X_flat / 255.0).astype(np.float32)

print("Matriz de características X:")
print("  Forma:", X.shape, "(filas = ejemplos, columnas = píxeles)")
print("  Tipo de dato:", X.dtype)
print("  Rango de valores: [%.2f, %.2f]" % (X.min(), X.max()))
print("\nVector objetivo y:")
print("  Forma:", y.shape)
print("  Clases únicas:", np.unique(y))
print("  Distribución total de clases:\n", pd.Series(y).value_counts().reindex(VOCALES).to_string())

### 11.6 Función de inspección y visualización inversa (`ver_vocal`)

De forma análoga a la función `ver_digito(ejemplo)` utilizada por la profesora en el notebook de referencia de MNIST, se implementa una función auxiliar para inspeccionar visualmente cualquier fila de la matriz de características $X$.

La función toma el vector aplanado de $7.500$ columnas, le aplica un redimensionamiento inverso (`.reshape(100, 75)`) y proyecta la imagen en escala de grises con su etiqueta de clase real, confirmando que la vectorización conserva fielmente la geometría original del carácter manuscrito.


In [ ]:
def ver_vocal(ejemplo, X_data=X, y_data=y, ancho=ANCHO, alto=ALTO):
    """Reconstruye y muestra la imagen de la vocal desde su vector plano de características."""
    if hasattr(X_data, "iloc"):
        vector = X_data.iloc[ejemplo].values
        etiqueta = y_data.iloc[ejemplo]
    else:
        vector = X_data[ejemplo]
        etiqueta = y_data[ejemplo]
    
    imagen = vector.reshape([alto, ancho])
    plt.figure(figsize=(2.6, 3.2))
    plt.imshow(imagen, cmap="gray", vmin=0.0, vmax=1.0)
    plt.title(f"Ejemplo: {ejemplo} | Vocal: {etiqueta}", fontsize=11)
    plt.axis("off")
    plt.tight_layout()
    plt.show()

# Visualizamos el primer ejemplo reconstruido directamente desde la matriz X
ver_vocal(0)

### 11.7 Partición estructurada sin fuga de información (*Data Leakage*)

En el notebook de referencia, la profesora establece una regla metodológica cardinal:
> *"Cualquier transformación de los datos, incluyendo PCA [o escalado], se debe ajustar únicamente en el conjunto de entrenamiento y luego aplicarse a los conjuntos de validación y prueba. Ajustar con todos los datos resultaría en lo que se llama data leakage."*

Adicionalmente, el dataset fue ampliado mediante aumento de datos (*data augmentation*), siguiendo la convención de nombres `aug<id>_<sufijo>`, donde un mismo trazo base (`aug<id>`) genera múltiples variantes.

Para evitar que variantes del mismo trazo se repartan entre entrenamiento y prueba (lo que causaría un sesgo optimista artificial), se implementa una partición agrupada y estratificada (`StratifiedGroupKFold`):
- **Entrenamiento:** $80\%$ de los grupos base ($4.056$ imágenes).
- **Prueba:** $20\%$ de los grupos base ($1.014$ imágenes), reservadas estrictamente para la evaluación final de todos los modelos.


In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

# Identificador del grupo base a partir del nombre del archivo (ej. 'aug1001_...')
base_id = [re.match(r"aug(\d+)_", Path(r).name).group(1) for r in df_bn["ruta_original"]]
grupos = np.array([f"{c}_{b}" for c, b in zip(df_bn["clase"], base_id)])

# Partición agrupada y estratificada (80% entrenamiento, 20% prueba)
particionador = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_indices, test_indices = next(particionador.split(X, y, grupos))

X_train, X_test = X[train_indices], X[test_indices]
y_train, y_test = y[train_indices], y[test_indices]
grupos_train, grupos_test = grupos[train_indices], grupos[test_indices]

# Creación de DataFrames con índices secuenciales para trazabilidad (estilo MNIST de la profesora)
train_index = range(0, len(X_train))
test_index = range(len(X_train), len(X_train) + len(X_test))

df_X_train = pd.DataFrame(data=X_train, index=train_index)
s_y_train = pd.Series(data=y_train, index=train_index, name="vocal")

df_X_test = pd.DataFrame(data=X_test, index=test_index)
s_y_test = pd.Series(data=y_test, index=test_index, name="vocal")

print("Dimensiones de las particiones creadas:")
print("  Entrenamiento: %d ejemplos con %d características" % X_train.shape)
print("  Prueba:        %d ejemplos con %d características" % X_test.shape)
print("  Grupos compartidos entre entrenamiento y prueba:", len(set(grupos_train) & set(grupos_test)))
print("\nDistribución de clases en el conjunto de prueba:")
print(pd.Series(y_test).value_counts().reindex(VOCALES).to_string())

### 11.8 Análisis de varianza de los píxeles (Redundancia de variables)

Al tratar cada píxel como una variable independiente, se dispone de $7.500$ variables. Siguiendo el análisis del cuaderno de la profesora sobre MNIST, muchas de estas columnas corresponden a bordes o esquinas de la imagen donde no hay trazo manuscrito (el valor de intensidad es casi idéntico en todas las muestras).

El cálculo de la varianza a lo largo del conjunto de entrenamiento permite identificar qué porcentaje de las variables aporta variación real y cuáles tienen varianza nula o casi nula.

In [ ]:
# Varianza de cada píxel a lo largo de los ejemplos de entrenamiento
varianza_pixeles = X_train.var(axis=0)

n_total = X_train.shape[1]
n_constantes = int((varianza_pixeles == 0).sum())
n_casi_constantes = int((varianza_pixeles < 0.005).sum())

print("Resumen de varianza de las variables (píxeles):")
print("  Número total de variables:             %d" % n_total)
print("  Variables constantes (varianza cero):  %d" % n_constantes)
print("  Variables con varianza menor a 0.005:  %d (%.1f%%)" % (n_casi_constantes, (n_casi_constantes / n_total) * 100))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

# Histograma de varianzas
conteos, bordes, _ = axes[0].hist(varianza_pixeles, bins=40, color="steelblue", edgecolor="black")
axes[0].set_xlabel("Varianza de la variable (píxel)")
axes[0].set_ylabel("Número de variables")
axes[0].set_title("Distribución de la varianza de las 7.500 variables")
axes[0].annotate(f"{conteos[0]:.0f} variables con\nvarianza casi nula",
                 xy=(bordes[1], conteos[0]),
                 xytext=(bordes[1] + 0.015, conteos[0] * 0.75),
                 arrowprops=dict(arrowstyle="->", color="red", lw=1.5))
axes[0].grid(True, axis="y", alpha=0.5)

# Mapa espacial 2D de la varianza por píxel
mapa_varianza = varianza_pixeles.reshape((ALTO, ANCHO))
im_var = axes[1].imshow(mapa_varianza, cmap="inferno")
axes[1].set_title("Mapa espacial de varianza por píxel\n(Zonas oscuras = bordes sin variación)")
axes[1].axis("off")
fig.colorbar(im_var, ax=axes[1], fraction=0.046, pad=0.04, label="Varianza")

plt.tight_layout()
plt.show()

### 11.9 Reducción de dimensionalidad con PCA (Análisis de Componentes Principales)

El Análisis de Componentes Principales (PCA) proyecta las $7.500$ variables originales sobre un nuevo conjunto de ejes ortogonales (componentes principales), ordenados de mayor a menor varianza explicada.

**Fundamentos metodológicos (aplicados según el cuaderno de la profesora):**
1. **Centrado:** `PCA` centra las variables automáticamente restando su media.
2. **Escalado:** Todas las variables comparten la misma escala continua $[0.0, 1.0]$.
3. **Prevención de fuga:** El modelo PCA se ajusta (`fit`) **únicamente sobre el conjunto de entrenamiento** (`X_train`). El conjunto de prueba (`X_test`) únicamente se proyecta (`transform`) usando el modelo ya ajustado.

Se evalúa la varianza explicada acumulada y se determina el número de componentes necesario para retener el **95 % de la varianza**.

In [ ]:
from sklearn.decomposition import PCA

# 1) Ajuste exploratorio de PCA en entrenamiento (300 componentes principales)
pca_exploratorio = PCA(n_components=300, svd_solver="randomized", random_state=42)
pca_exploratorio.fit(X_train)

varianza_acumulada = np.cumsum(pca_exploratorio.explained_variance_ratio_)

plt.figure(figsize=(8, 4.5))
plt.plot(range(1, len(varianza_acumulada) + 1), varianza_acumulada, lw=2, color="royalblue")
plt.axhline(y=0.90, color="orange", linestyle=":", label="90% de varianza explicada")
plt.axhline(y=0.95, color="red", linestyle="--", label="95% de varianza explicada")
plt.xlabel("Número de componentes principales (k)")
plt.ylabel("Varianza explicada acumulada")
plt.title("Varianza explicada acumulada según el número de componentes (PCA)")
plt.legend()
plt.grid(True, alpha=0.5)
plt.tight_layout()
plt.show()

# Identificación de componentes necesarias para umbrales clave
k_90 = int(np.searchsorted(varianza_acumulada, 0.90) + 1)
k_95 = int(np.searchsorted(varianza_acumulada, 0.95) + 1)
print(f"Componentes necesarias para el 90% de la varianza: k = {k_90}")
print(f"Componentes necesarias para el 95% de la varianza: k = {k_95} (de 7.500 iniciales, reducción del {(1 - k_95/7500)*100:.1f}%)")

# 2) Ajuste del PCA definitivo para el 95% de varianza
pca_95 = PCA(n_components=k_95, svd_solver="randomized", random_state=42)
X_train_pca = pca_95.fit_transform(X_train)
X_test_pca = pca_95.transform(X_test)  # Sin fit: estrictamente transform para evitar data leakage

print("\nDimensiones de los datos reducidos con PCA (95% de varianza):")
print("  X_train_pca:", X_train_pca.shape)
print("  X_test_pca: ", X_test_pca.shape)

# 3) Visualización en 2D: Primeras 2 componentes de PCA vs. 2 píxeles individuales
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for vocal in VOCALES:
    mask = (y_train == vocal)
    axes[0].scatter(X_train_pca[mask, 0], X_train_pca[mask, 1], label=vocal, alpha=0.5, s=15)
axes[0].set_title("Separación de vocales con las 2 primeras componentes de PCA", fontsize=10)
axes[0].set_xlabel("Componente Principal 1")
axes[0].set_ylabel("Componente Principal 2")
axes[0].legend(title="Vocal")
axes[0].grid(True, alpha=0.4)

p1, p2 = 3785, 3795
for vocal in VOCALES:
    mask = (y_train == vocal)
    axes[1].scatter(X_train[mask, p1], X_train[mask, p2], label=vocal, alpha=0.5, s=15)
axes[1].set_title(f"Dispersión con 2 píxeles individuales (columnas {p1} y {p2})", fontsize=10)
axes[1].set_xlabel(f"Intensidad Píxel {p1}")
axes[1].set_ylabel(f"Intensidad Píxel {p2}")
axes[1].legend(title="Vocal")
axes[1].grid(True, alpha=0.4)

plt.tight_layout()
plt.show()

# 4) Reconstrucción visual de una vocal según distintos números de componentes k
ejemplo_idx = 0
valores_k = [10, 50, 100, k_95, 300]
z_ejemplo = pca_exploratorio.transform(X_train[[ejemplo_idx]])[0]

fig, ejes = plt.subplots(1, len(valores_k) + 1, figsize=(15, 3.2))
ejes[0].imshow(X_train[ejemplo_idx].reshape(ALTO, ANCHO), cmap="gray", vmin=0.0, vmax=1.0)
ejes[0].set_title(f"Original ({y_train[ejemplo_idx]})", fontsize=10)
ejes[0].axis("off")

for eje, k in zip(ejes[1:], valores_k):
    x_reconstruido = pca_exploratorio.mean_ + z_ejemplo[:k] @ pca_exploratorio.components_[:k, :]
    eje.imshow(x_reconstruido.reshape(ALTO, ANCHO), cmap="gray", vmin=0.0, vmax=1.0)
    eje.set_title(f"k = {k}", fontsize=10)
    eje.axis("off")

plt.suptitle("Reconstrucción de una vocal según el número de componentes principales (k)", y=1.03)
plt.tight_layout()
plt.show()


### 11.10 Estandarización y contrato de datos para el equipo

Con esta fase completada, los datos quedan estructurados de forma homogénea para la implementación y ajuste de los métodos respectivos con `GridSearchCV`:

**Variables y objetos disponibles en memoria para todos los modelos:**
- `X_train`, `y_train`: Datos de entrenamiento normalizados $[0.0, 1.0]$ en el espacio original de $7.500$ píxeles ($4.056$ ejemplos).
- `X_test`, `y_test`: Datos de prueba reservados para evaluación final ($1.014$ ejemplos).
- `X_train_pca`, `X_test_pca`: Datos transformados con PCA conservando el $95\%$ de varianza ($k = 135$ variables).
- `grupos_train`: Vector de grupos de trazo base para configurar la validación cruzada agrupada sin sesgo por aumento.
- `ver_vocal(idx)`: Función para inspección visual de cualquier ejemplo o predicción.


In [ ]:
# Verificación final de los objetos compartidos para todos los métodos
print("=== CONTRATO DE DATOS PREPARADOS PARA EL EQUIPO ===")
print(f"1. Espacio Original:   X_train = {X_train.shape} | X_test = {X_test.shape} (tipo: {X_train.dtype})")
print(f"2. Espacio PCA (95%):  X_train_pca = {X_train_pca.shape} | X_test_pca = {X_test_pca.shape} (tipo: {X_train_pca.dtype})")
print(f"3. Etiquetas:          y_train = {y_train.shape} | y_test = {y_test.shape} (Clases: {np.unique(y_train)})")
print(f"4. Grupos (sin fuga):  grupos_train = {grupos_train.shape} | grupos únicos = {len(np.unique(grupos_train))}")

## 12. Árboles de decisión

Un árbol de decisión clasifica mediante una secuencia de reglas binarias sobre las variables de entrada. En cada nodo se elige el píxel y el umbral que más reducen la impureza de las clases; el crecimiento se detiene por profundidad, tamaño mínimo de hoja o pureza. La impureza se mide con **Gini** o **entropía**. El método no requiere escalado y es interpretable, pero puede sobreajustar si se deja crecer sin control.

### Referencia metodológica

Como referencia se revisó el repositorio de ejemplos de la profesora:

- [`tree/diabetes_indian_parte1_IA.ipynb`](https://github.com/idanis/IA_Unimagdalena/blob/main/tree/diabetes_indian_parte1_IA.ipynb): usa `DecisionTreeClassifier`, `criterion='entropy'`, `splitter='best'`, y evalúa con accuracy, matriz de confusión y `classification_report`.
- [`Random_Forest/regression_tree_example _ver2.ipynb`](https://github.com/idanis/IA_Unimagdalena/blob/main/Random_Forest/regression_tree_example%20_ver2.ipynb): ilustra `GridSearchCV` con `cv=5`, `best_params_` y evaluación final sobre el conjunto de prueba.

Este trabajo sigue esa estructura general y la adapta al dataset de imágenes: agrega binarización con Otsu, partición agrupada por aumento y métricas adicionales como balanced accuracy y F1 macro.

### 12.1 Diseño del experimento

Se ajustan los hiperparámetros con `GridSearchCV` sobre el conjunto de entrenamiento, usando validación cruzada agrupada y estratificada de 5 pliegues (`StratifiedGroupKFold`). El conjunto de prueba no participa en la búsqueda. La métrica de selección es *accuracy* y se registra también el desempeño en entrenamiento para detectar sobreajuste.

| Hiperparámetro | Valores evaluados | Efecto |
|---|---|---|
| `criterion` | `gini`, `entropy` | Medida de impureza de los nodos |
| `max_depth` | 6, 10, 14, sin límite | Controla la complejidad y el sobreajuste |
| `min_samples_leaf` | 1, 3 | Evita hojas con muy pocos casos |
| `max_features` | todas, `sqrt` | Número de píxeles considerados en cada división |


In [ ]:
parametros_arbol = {
    "criterion": ["gini", "entropy"],
    "max_depth": [6, 10, 14, None],
    "min_samples_leaf": [1, 3],
    "max_features": [None, "sqrt"],
}

cv_arbol = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
busqueda_arbol = GridSearchCV(
    estimator=DecisionTreeClassifier(random_state=42),
    param_grid=parametros_arbol,
    scoring="accuracy",
    cv=cv_arbol,
    n_jobs=-1,
    return_train_score=True,
    refit=True,
)

inicio = time.time()
busqueda_arbol.fit(X_train, y_train, groups=grupos_train)
duracion_busqueda = time.time() - inicio

print(f"Combinaciones evaluadas: {len(busqueda_arbol.cv_results_['params'])}")
print(f"Tiempo de búsqueda: {duracion_busqueda:.1f} s")
print("Mejores hiperparámetros:")
for clave, valor in busqueda_arbol.best_params_.items():
    print(f"  {clave}: {valor}")
print(f"Mejor accuracy de validación cruzada: {busqueda_arbol.best_score_:.4f}")


In [ ]:
resultados_arbol = pd.DataFrame(busqueda_arbol.cv_results_)
resultados_arbol["profundidad"] = resultados_arbol["param_max_depth"].apply(
    lambda valor: "sin límite" if pd.isna(valor) else str(valor)
)
resultados_arbol["variables"] = resultados_arbol["param_max_features"].apply(
    lambda valor: "todas" if pd.isna(valor) else str(valor)
)
columnas = ["rank_test_score", "param_criterion", "param_max_depth",
            "param_min_samples_leaf", "variables",
            "mean_train_score", "mean_test_score", "std_test_score"]
tabla_arbol = resultados_arbol.sort_values("rank_test_score")[columnas].head(10).copy()
for columna in ["mean_train_score", "mean_test_score", "std_test_score"]:
    tabla_arbol[columna] = tabla_arbol[columna].round(4)
tabla_arbol


In [ ]:
# Efecto de la profundidad y de max_features en la validación cruzada
resumen_cv = (resultados_arbol
              .groupby(["profundidad", "variables"])["mean_test_score"]
              .mean()
              .reset_index())
orden = ["6", "10", "14", "sin límite"]
fig, ax = plt.subplots(figsize=(8, 4))
for valor, grupo in resumen_cv.groupby("variables"):
    etiqueta = "todas las variables" if valor == "todas" else f"max_features={valor}"
    datos = grupo.set_index("profundidad").reindex(orden)
    ax.plot(orden, datos["mean_test_score"], marker="o", label=etiqueta)
ax.set_title("Accuracy promedio de validación cruzada según la profundidad")
ax.set_xlabel("Profundidad máxima del árbol")
ax.set_ylabel("Accuracy promedio (5 pliegues)")
ax.set_ylim(0.9, 1.005)
ax.grid(alpha=0.3)
ax.legend(title="Variables por división")
plt.tight_layout()
plt.show()


### 12.2 Evaluación sobre el conjunto de prueba

Se evalúa el mejor modelo una sola vez sobre las 1.014 imágenes reservadas. Se reportan *accuracy*, *balanced accuracy*, precisión, recall y F1 por clase, además de la matriz de confusión. Como el problema es de clasificación multiclase, las métricas de regresión (MSE, coeficiente de determinación y MAPE) no aplican; esas métricas corresponden a los métodos de regresión de la guía.


In [ ]:
mejor_arbol = busqueda_arbol.best_estimator_
y_pred_arbol = mejor_arbol.predict(X_test)

accuracy_prueba = accuracy_score(y_test, y_pred_arbol)
accuracy_entrenamiento = accuracy_score(y_train, mejor_arbol.predict(X_train))
balanced_accuracy_trees = balanced_accuracy_score(y_test, y_pred_arbol)

print(f"Accuracy en entrenamiento: {accuracy_entrenamiento:.4f}")
print(f"Accuracy en prueba:        {accuracy_prueba:.4f}")
print(f"Balanced accuracy:         {balanced_accuracy_trees:.4f}")
print(f"Profundidad del árbol:     {mejor_arbol.get_depth()}")
print(f"Número de hojas:           {mejor_arbol.get_n_leaves()}")
print("\nReporte por clase:")
print(classification_report(y_test, y_pred_arbol, labels=CLASES, digits=4))


In [ ]:
# Matriz de confusión: conteos y proporciones por clase real
matriz = confusion_matrix(y_test, y_pred_arbol, labels=CLASES)
matriz_norm = matriz / matriz.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, datos, titulo, formato in [
    (axes[0], matriz, "Conteos", "d"),
    (axes[1], matriz_norm, "Proporción por clase real", ".2f"),
]:
    imagen = ax.imshow(datos, cmap="Blues", vmin=0, vmax=datos.max())
    ax.set_xticks(range(len(CLASES)))
    ax.set_yticks(range(len(CLASES)))
    ax.set_xticklabels(CLASES)
    ax.set_yticklabels(CLASES)
    ax.set_xlabel("Clase predicha")
    ax.set_ylabel("Clase real")
    ax.set_title(titulo)
    for i in range(len(CLASES)):
        for j in range(len(CLASES)):
            valor = datos[i, j]
            texto = f"{valor:{formato}}"
            ax.text(j, i, texto, ha="center", va="center",
                    color="white" if valor > datos.max() / 2 else "black")
    fig.colorbar(imagen, ax=ax, fraction=0.046, pad=0.04)
plt.suptitle("Matriz de confusión del árbol de decisión ajustado")
plt.tight_layout()
plt.show()


In [ ]:
# Importancia de los píxeles utilizados por el árbol
importancias = mejor_arbol.feature_importances_.reshape(ALTO, ANCHO)
promedio = X_train.mean(axis=0).reshape(ALTO, ANCHO)

fig, axes = plt.subplots(1, 2, figsize=(10, 6))
axes[0].imshow(promedio, cmap="gray_r")
axes[0].set_title("Imagen promedio de entrenamiento")
axes[0].set_xlabel("Columna (0-74)")
axes[0].set_ylabel("Fila (0-99)")
axes[1].imshow(promedio, cmap="gray_r")
imagen = axes[1].imshow(importancias, cmap="magma", alpha=0.75)
axes[1].set_title("Píxeles usados por el árbol")
axes[1].set_xlabel("Columna (0-74)")
axes[1].set_ylabel("Fila (0-99)")
fig.colorbar(imagen, ax=axes[1], shrink=0.8, label="Importancia")
plt.tight_layout()
plt.show()

print("Píxeles más importantes (fila, columna, importancia):")
for indice in np.argsort(importancias.ravel())[::-1][:10]:
    fila, columna = np.unravel_index(indice, importancias.shape)
    print(f"  ({fila}, {columna}): {importancias[fila, columna]:.4f}")
print(f"Píxeles con importancia mayor que cero: {np.count_nonzero(importancias)} de {importancias.size}")


### 12.3 Análisis del experimento


In [ ]:
# Resultados generales
mejor_accuracy_cv = busqueda_arbol.best_score_
diferencia_accuracy = accuracy_entrenamiento - accuracy_prueba
errores_totales = int(np.sum(matriz) - np.trace(matriz))
pixeles_utilizados = int(np.count_nonzero(importancias))

# Configuraciones que alcanzaron el mejor accuracy de validación cruzada
mejor_puntuacion = resultados_arbol["mean_test_score"].max()
configuraciones_empatadas = resultados_arbol[
    np.isclose(resultados_arbol["mean_test_score"], mejor_puntuacion)
]

# Recall por clase
reporte = classification_report(
    y_test, y_pred_arbol,
    labels=CLASES,
    output_dict=True,
    zero_division=0
)

recalls = {
    str(clase): reporte[str(clase)]["recall"]
    for clase in CLASES
}

clase_mayor_recall = max(recalls, key=recalls.get)
clase_menor_recall = min(recalls, key=recalls.get)

# Errores de clasificación más frecuentes
errores_por_clase = []

for i, j in np.argwhere(matriz > 0):
    if i != j:
        errores_por_clase.append(
            (int(matriz[i, j]), CLASES[i], CLASES[j])
        )

errores_por_clase.sort(reverse=True)

print("ANÁLISIS DEL EXPERIMENTO")
print("=" * 60)

print(
    f"El mejor árbol obtuvo un accuracy de validación cruzada "
    f"de {mejor_accuracy_cv:.2%}."
)

print(
    f"Los mejores hiperparámetros fueron: "
    f"{busqueda_arbol.best_params_}."
)

print(
    f"Se encontraron {len(configuraciones_empatadas)} "
    f"configuraciones con el mejor accuracy de validación cruzada."
)

print("\nDesempeño:")
print(f"- Accuracy de entrenamiento: {accuracy_entrenamiento:.2%}")
print(f"- Accuracy de prueba: {accuracy_prueba:.2%}")
print(f"- Balanced accuracy: {balanced_accuracy_trees:.2%}")
print(f"- Diferencia entrenamiento-prueba: {diferencia_accuracy:.2%}")

print("\nComplejidad del árbol:")
print(f"- Profundidad: {mejor_arbol.get_depth()}")
print(f"- Número de hojas: {mejor_arbol.get_n_leaves()}")
print(f"- Píxeles con importancia distinta de cero: "
      f"{pixeles_utilizados} de {importancias.size}")

print("\nMatriz de confusión:")
print(f"- Total de errores: {errores_totales}")

for cantidad, real, predicha in errores_por_clase:
    print(f"- {real} clasificadas como {predicha}: {cantidad}")

print("\nRecall por clase:")
for clase, recall in recalls.items():
    print(f"- {clase}: {recall:.2%}")

print(
    f"\nMayor recall: {clase_mayor_recall} "
    f"({recalls[clase_mayor_recall]:.2%})"
)
print(
    f"Menor recall: {clase_menor_recall} "
    f"({recalls[clase_menor_recall]:.2%})"
)

## 13. Comparación entre los modelos

Esta rama del trabajo se enfoca en **árboles de decisión**. La comparación con los otros métodos de la guía (regresión multivariada, random forest, MLP y DNN) se integra en la versión final del equipo; aquí se comparan variantes del árbol para aislar el efecto de los hiperparámetros y de la representación de los datos.

Para decidir si el ajuste de hiperparámetros aporta una mejora real, se comparan cuatro alternativas sobre el mismo conjunto de prueba:

1. **Línea base:** predice siempre la clase más frecuente. Sirve para verificar que el modelo aprende algo más que el desbalance.
2. **Árbol sin ajuste:** hiperparámetros por defecto, sin límite de profundidad.
3. **Árbol ajustado:** mejores hiperparámetros encontrados por `GridSearchCV`, con los 7.500 píxeles.
4. **Árbol ajustado con reducción:** los mismos hiperparámetros sobre una versión reducida a 20x25 píxeles (500 variables), para medir el costo de conservar la resolución completa.

Todas las variantes usan la misma partición agrupada y se evalúan con accuracy, balanced accuracy y F1 macro.


In [ ]:
def reducir_binarizada(matriz, ancho=20, alto=25):
    """Reduce cada imagen binaria a una cuadrícula de ancho x alto."""
    return np.stack([
        (np.asarray(Image.fromarray(fila.reshape(ALTO, ANCHO) * 255)
                    .resize((ancho, alto), Image.Resampling.BOX)) > 127).astype(np.uint8).ravel()
        for fila in matriz
    ])

X_train_red = reducir_binarizada(X_train)
X_test_red = reducir_binarizada(X_test)

configuraciones = {
    "Línea base (clase mayoritaria)": (
        DummyClassifier(strategy="most_frequent"), X_train, X_test),
    "Árbol sin ajuste": (
        DecisionTreeClassifier(random_state=42), X_train, X_test),
    "Árbol ajustado (7.500 píxeles)": (
        DecisionTreeClassifier(random_state=42, **busqueda_arbol.best_params_), X_train, X_test),
    "Árbol ajustado (500 píxeles)": (
        DecisionTreeClassifier(random_state=42, **busqueda_arbol.best_params_), X_train_red, X_test_red),
}

filas_comparacion = []
for nombre, (modelo, Xe, Xp) in configuraciones.items():
    inicio = time.time()
    modelo.fit(Xe, y_train)
    duracion = time.time() - inicio
    y_pred_modelo = modelo.predict(Xp)
    filas_comparacion.append({
        "Modelo": nombre,
        "Accuracy entrenamiento": accuracy_score(y_train, modelo.predict(Xe)),
        "Accuracy prueba": accuracy_score(y_test, y_pred_modelo),
        "Balanced accuracy": balanced_accuracy_score(y_test, y_pred_modelo),
        "F1 macro": f1_score(y_test, y_pred_modelo, average="macro"),
        "Profundidad": getattr(modelo, "get_depth", lambda: None)(),
        "Hojas": getattr(modelo, "get_n_leaves", lambda: None)(),
        "Tiempo de ajuste (s)": duracion,
    })

comparacion = pd.DataFrame(filas_comparacion)
comparacion.round(4)


In [ ]:
# Comparación visual de accuracy en entrenamiento y prueba
posiciones = np.arange(len(comparacion))
ancho = 0.38

fig, ax = plt.subplots(figsize=(9.5, 4.5))
ax.bar(posiciones - ancho / 2, comparacion["Accuracy entrenamiento"],
       width=ancho, label="Entrenamiento", color="steelblue")
ax.bar(posiciones + ancho / 2, comparacion["Accuracy prueba"],
       width=ancho, label="Prueba", color="darkorange")
ax.set_xticks(posiciones)
ax.set_xticklabels(comparacion["Modelo"], rotation=12, ha="right")
ax.set_ylim(0, 1.08)
ax.set_ylabel("Accuracy")
ax.set_title("Comparación de los modelos de árbol de decisión")
ax.axhline(0.2, color="gray", linestyle="--", linewidth=1, label="Azar (5 clases)")
for posicion, valor in zip(posiciones + ancho / 2, comparacion["Accuracy prueba"]):
    ax.text(posicion, valor + 0.02, f"{valor:.3f}", ha="center", fontsize=9)
ax.legend(loc="lower right")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


### 13.1 Lectura de la comparación

In [ ]:
# Recuperar los resultados de cada modelo
resultados = comparacion.set_index("Modelo")

base = resultados.loc["Línea base (clase mayoritaria)"]
arbol_completo = resultados.loc["Árbol ajustado (7.500 píxeles)"]
arbol_reducido = resultados.loc["Árbol ajustado (500 píxeles)"]

# Obtener el modelo ajustado con 7.500 píxeles
modelo_completo = configuraciones[
    "Árbol ajustado (7.500 píxeles)"
][0]

# Calcular diferencias de accuracy en puntos porcentuales
diferencia_base = (
    arbol_completo["Accuracy prueba"] - base["Accuracy prueba"]
) * 100

diferencia_reduccion = (
    arbol_completo["Accuracy prueba"] - arbol_reducido["Accuracy prueba"]
) * 100

# Calcular el factor de aceleración
if arbol_reducido["Tiempo de ajuste (s)"] > 0:
    factor_aceleracion = (
        arbol_completo["Tiempo de ajuste (s)"]
        / arbol_reducido["Tiempo de ajuste (s)"]
    )
else:
    factor_aceleracion = None

# Obtener la complejidad y las características del árbol
numero_hojas = modelo_completo.get_n_leaves()
pixeles_importantes = np.count_nonzero(
    modelo_completo.feature_importances_ > 0
)

print("LECTURA DE LA COMPARACIÓN\n")

print(
    f"La línea base, que predice siempre la clase mayoritaria, "
    f"obtiene un accuracy de {base['Accuracy prueba']:.2%}. "
    f"El árbol ajustado con 7.500 píxeles alcanza "
    f"{arbol_completo['Accuracy prueba']:.2%}, "
    f"una diferencia de {diferencia_base:.2f} puntos porcentuales."
)

print(
    f"\nEl árbol ajustado con 7.500 píxeles obtiene un accuracy "
    f"de {arbol_completo['Accuracy prueba']:.2%}, mientras que "
    f"la versión reducida a 500 píxeles obtiene "
    f"{arbol_reducido['Accuracy prueba']:.2%}. "
    f"La diferencia es de {diferencia_reduccion:.2f} puntos porcentuales."
)

print(
    f"El tiempo de ajuste pasa de "
    f"{arbol_completo['Tiempo de ajuste (s)']:.4f} s "
    f"a {arbol_reducido['Tiempo de ajuste (s)']:.4f} s."
)

if factor_aceleracion is not None:
    print(
        f"Esto representa un factor de aceleración de "
        f"{factor_aceleracion:.2f} veces."
    )

print(
    f"\nEl árbol ajustado con 7.500 píxeles tiene "
    f"{numero_hojas} hojas y utiliza "
    f"{pixeles_importantes} píxeles con importancia mayor que cero."
)

### Referencias

- Kaggle. *Handwriting Letter and Number Dataset*. https://www.kaggle.com/datasets/straighttoheaven/handwriting-letter-and-number-dataset
- Otsu, N. (1979). A threshold selection method from gray-level histograms. *IEEE Transactions on Systems, Man, and Cybernetics*, 9(1), 62-66.
- Breiman, L., Friedman, J., Stone, C. J., & Olshen, R. A. (1984). *Classification and Regression Trees*. Wadsworth.
- scikit-learn. *Decision Trees* y *GridSearchCV*. https://scikit-learn.org/stable/modules/tree.html


## Random Forest

Random Forest combina las predicciones de varios árboles de decisión para clasificar las imágenes de las vocales A, E, I, O y U. Se compara una configuración base con un modelo ajustado mediante GridSearchCV.

### Modelo base

Se utiliza `max_depth=3`, `n_estimators=50`, `criterion="gini"` y `random_state=42`. El modelo se entrena con las características compartidas de `X_train` y las etiquetas de `y_train`, conservando la preparación y la partición común del equipo. No se añade preprocesamiento ni se utilizan las matrices PCA.

La evaluación incluye accuracy de entrenamiento y prueba, matriz de confusión e informe de clasificación.


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import plot_tree
from sklearn.metrics import confusion_matrix, classification_report

modelo_rf = RandomForestClassifier(
    max_depth=3,
    n_estimators=50,
    criterion="gini",
    random_state=42
)
modelo_rf.fit(X_train, y_train)


In [ ]:
y_pred_train_rf = modelo_rf.predict(X_train)
y_pred_test_rf = modelo_rf.predict(X_test)

print("Accuracy de entrenamiento:", modelo_rf.score(X_train, y_train))
print("Accuracy de prueba:", modelo_rf.score(X_test, y_test))


In [ ]:
matriz_confusion_rf = confusion_matrix(
    y_test, y_pred_test_rf, labels=modelo_rf.classes_
)
print("Matriz de confusión de prueba (filas: reales; columnas: predichas):")
pd.DataFrame(
    matriz_confusion_rf,
    index=modelo_rf.classes_,
    columns=modelo_rf.classes_
)


In [ ]:
print("Reporte de clasificación del conjunto de prueba:")
print(classification_report(
    y_test,
    y_pred_test_rf,
    labels=modelo_rf.classes_,
    target_names=modelo_rf.classes_
))


#### Visualización de un árbol

Se representa el primer árbol del bosque. Las variables `x[índice]` corresponden a los píxeles de entrada; las métricas anteriores pertenecen al modelo completo.


In [ ]:
arbol_rf = modelo_rf.estimators_[0]
plt.figure(figsize=(22, 9))
plot_tree(
    arbol_rf,
    class_names=modelo_rf.classes_,
    filled=True,
    fontsize=8
)
plt.title("Random Forest — Primer árbol del modelo base")
plt.tight_layout()
plt.show()


### Ajuste mediante GridSearchCV

Se evalúa la grilla trabajada en clase para seleccionar la configuración con mayor accuracy medio de validación cruzada. Se conserva `return_train_score=True`.


In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid_rf = {
    'max_depth': [3, 4, 5],
    'criterion': ['gini', 'entropy'],
    'n_estimators': [50, 100, 150],
    'min_samples_leaf': [1, 2, 3],
    'min_samples_split': [2, 3, 4]
}


In [ ]:
cv_rf = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


In [ ]:
rf_gs = RandomForestClassifier(random_state=42, n_jobs=-1)

grid_search_rf = GridSearchCV(
    rf_gs,
    param_grid_rf,
    scoring="accuracy",
    cv=cv_rf,
    return_train_score=True,
    n_jobs=1
)


La búsqueda evalúa **162 configuraciones** con **cinco pliegues estratificados y agrupados**, utilizando únicamente `X_train`, `y_train` y `grupos_train`. El conjunto de prueba se reserva para evaluar el modelo después de la selección.

Se realizan 810 ajustes de validación y un reajuste de la mejor configuración sobre todo el conjunto de entrenamiento.


In [ ]:
grid_search_rf.fit(X_train, y_train, groups=grupos_train)


### Mejores hiperparámetros y análisis de validación

Los parámetros y scores siguientes corresponden a la búsqueda de esta ejecución. La tabla resume las diez configuraciones con mejor ranking; los scores se expresan entre 0 y 1.


In [ ]:
print("Mejores hiperparámetros:", grid_search_rf.best_params_)
print("Mejor accuracy medio de validación cruzada:", grid_search_rf.best_score_)

mejor_modelo_rf = grid_search_rf.best_estimator_
print("Mejor estimador:", mejor_modelo_rf)


In [ ]:
resultados_cv_rf = pd.DataFrame(grid_search_rf.cv_results_)
columnas_cv_rf = [
    "rank_test_score", "param_criterion", "param_max_depth", "param_n_estimators",
    "param_min_samples_leaf", "param_min_samples_split",
    "mean_train_score", "mean_test_score", "std_test_score"
]
mejores_configuraciones_rf = (
    resultados_cv_rf.sort_values("rank_test_score", kind="stable")
    .head(10)[columnas_cv_rf]
    .rename(columns={
        "param_criterion": "criterion", "param_max_depth": "max_depth",
        "param_n_estimators": "n_estimators", "param_min_samples_leaf": "min_samples_leaf",
        "param_min_samples_split": "min_samples_split"
    })
    .round(4)
)
indice_mejor_rf = grid_search_rf.best_index_
media_validacion_rf = resultados_cv_rf.loc[indice_mejor_rf, "mean_test_score"]
desviacion_validacion_rf = resultados_cv_rf.loc[indice_mejor_rf, "std_test_score"]
print(f"Accuracy medio de validación seleccionado: {media_validacion_rf:.4%}")
print(f"Desviación estándar: {desviacion_validacion_rf * 100:.4f} puntos porcentuales")
mejores_configuraciones_rf


Las primeras filas permiten comparar medias de entrenamiento y validación junto con la variabilidad entre pliegues. El redondeo puede mostrar medias iguales para configuraciones con distinto ranking; se conserva el ranking original y la selección no utiliza el conjunto de prueba.


In [ ]:
validacion_profundidad_rf = (
    resultados_cv_rf.groupby(["param_max_depth", "param_criterion"])["mean_test_score"]
    .mean().unstack("param_criterion").sort_index()
)
fig_rf, ax_rf = plt.subplots(figsize=(9, 5))
for criterio_rf, color_rf, marcador_rf, linea_rf in [
    ("gini", "steelblue", "o", "-"), ("entropy", "darkorange", "s", "--")
]:
    valores_cv_rf = validacion_profundidad_rf[criterio_rf] * 100
    ax_rf.plot(valores_cv_rf.index, valores_cv_rf, label=criterio_rf,
            color=color_rf, marker=marcador_rf, linestyle=linea_rf)
    print(f"{criterio_rf}: promedio con profundidad 3 = {valores_cv_rf.loc[3]:.4f} %, "
          f"con profundidad 5 = {valores_cv_rf.loc[5]:.4f} %")
ax_rf.set_title("Validación por profundidad y criterio\nPromedio de configuraciones de la grilla")
ax_rf.set_xlabel("Profundidad máxima")
ax_rf.set_ylabel("Accuracy medio de validación (%)")
ax_rf.set_xticks(validacion_profundidad_rf.index)
ax_rf.legend(title="Criterio")
ax_rf.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


Cada punto promedia los accuracies medios de validación de 27 configuraciones: se mantienen profundidad y criterio, mientras varían número de árboles, tamaño mínimo de hoja y tamaño mínimo de división. La curva describe ese promedio de la grilla; no representa una configuración individual ni aísla causalmente el efecto de la profundidad.


### Evaluación del modelo ajustado

Con la configuración seleccionada y reajustada sobre entrenamiento, se obtienen las predicciones, el accuracy, la matriz de confusión y el informe de clasificación del conjunto de prueba.


In [ ]:
y_pred_train_rf_ajustado = mejor_modelo_rf.predict(X_train)
y_pred_test_rf_ajustado = mejor_modelo_rf.predict(X_test)

accuracy_train_rf_ajustado = mejor_modelo_rf.score(X_train, y_train)
accuracy_test_rf_ajustado = mejor_modelo_rf.score(X_test, y_test)

print("Accuracy de entrenamiento del modelo ajustado:", accuracy_train_rf_ajustado)
print("Accuracy de prueba del modelo ajustado:", accuracy_test_rf_ajustado)


In [ ]:
matriz_confusion_rf_ajustado = confusion_matrix(
    y_test, y_pred_test_rf_ajustado, labels=mejor_modelo_rf.classes_
)
print("Matriz de confusión de prueba del modelo ajustado (filas: reales; columnas: predichas):")
pd.DataFrame(
    matriz_confusion_rf_ajustado,
    index=mejor_modelo_rf.classes_,
    columns=mejor_modelo_rf.classes_
)


In [ ]:
clases_rf = list(modelo_rf.classes_)
assert clases_rf == ["A", "E", "I", "O", "U"]
assert list(mejor_modelo_rf.classes_) == clases_rf
maximo_conteo_rf = max(matriz_confusion_rf.max(), matriz_confusion_rf_ajustado.max())
fig_rf, ejes_rf = plt.subplots(2, 2, figsize=(12, 10), constrained_layout=True)
for fila_rf, (nombre_rf, matriz_rf) in enumerate([
    ("Base", matriz_confusion_rf), ("Ajustado", matriz_confusion_rf_ajustado)
]):
    assert matriz_rf.shape == (len(clases_rf), len(clases_rf))
    totales_rf = matriz_rf.sum(axis=1, keepdims=True)
    normalizada_rf = np.divide(matriz_rf, totales_rf,
                               out=np.zeros_like(matriz_rf, dtype=float), where=totales_rf != 0)
    for columna_rf, (valores_rf, titulo_rf, limite_rf) in enumerate([
        (matriz_rf, "Conteos", maximo_conteo_rf),
        (normalizada_rf, "Proporción por clase real", 1)
    ]):
        ax_rf = ejes_rf[fila_rf, columna_rf]
        imagen_rf = ax_rf.imshow(valores_rf, cmap="Blues", vmin=0, vmax=limite_rf)
        ax_rf.set_title(f"{nombre_rf} — {titulo_rf}")
        ax_rf.set_xticks(range(len(clases_rf)), clases_rf)
        ax_rf.set_yticks(range(len(clases_rf)), clases_rf)
        ax_rf.set_xlabel("Vocal predicha")
        ax_rf.set_ylabel("Vocal real")
        for i_rf in range(len(clases_rf)):
            for j_rf in range(len(clases_rf)):
                etiqueta_rf = (str(int(valores_rf[i_rf, j_rf])) if columna_rf == 0
                               else f"{valores_rf[i_rf, j_rf]:.1%}")
                ax_rf.text(j_rf, i_rf, etiqueta_rf, ha="center", va="center",
                        color="white" if valores_rf[i_rf, j_rf] > limite_rf / 2 else "black")
        fig_rf.colorbar(imagen_rf, ax=ax_rf, label="Conteo" if columna_rf == 0 else "Proporción")
plt.show()


Los conteos muestran los errores absolutos y las proporciones permiten comparar las clases considerando sus distintos soportes. Las celdas O → U y U → O ayudan a localizar la confusión entre estas vocales; el resumen final calcula sus cambios con las matrices de la ejecución.


In [ ]:
print("Reporte de clasificación de prueba del modelo ajustado:")
print(classification_report(
    y_test,
    y_pred_test_rf_ajustado,
    labels=mejor_modelo_rf.classes_,
    target_names=mejor_modelo_rf.classes_
))


In [ ]:
reporte_rf_base = classification_report(
    y_test, y_pred_test_rf, labels=clases_rf, target_names=clases_rf, output_dict=True
)
reporte_rf_ajustado = classification_report(
    y_test, y_pred_test_rf_ajustado, labels=clases_rf, target_names=clases_rf, output_dict=True
)
tabla_metricas_rf = pd.DataFrame(reporte_rf_ajustado).T.loc[
    clases_rf, ["precision", "recall", "f1-score", "support"]
]
tabla_metricas_rf["support"] = tabla_metricas_rf["support"].astype(int)
vocal_precision_min_rf = tabla_metricas_rf["precision"].idxmin()
vocal_recall_min_rf = tabla_metricas_rf["recall"].idxmin()
print(f"Menor precision ajustada: {vocal_precision_min_rf} "
      f"({tabla_metricas_rf.loc[vocal_precision_min_rf, 'precision']:.2%})")
print(f"Menor recall ajustado: {vocal_recall_min_rf} "
      f"({tabla_metricas_rf.loc[vocal_recall_min_rf, 'recall']:.2%})")
tabla_metricas_rf.round(4)


Leer precision y recall conjuntamente permite distinguir falsas asignaciones a una vocal de ejemplos de esa vocal que el modelo omite. Las clases con menor valor se identifican arriba a partir del reporte calculado, sin fijarlas manualmente.


In [ ]:
tabla_recall_rf = pd.DataFrame({
    "Base": [reporte_rf_base[vocal_rf]["recall"] for vocal_rf in clases_rf],
    "Ajustado": [reporte_rf_ajustado[vocal_rf]["recall"] for vocal_rf in clases_rf]
}, index=clases_rf)
fig_rf, ax_rf = plt.subplots(figsize=(11, 5))
(tabla_recall_rf * 100).plot.bar(ax=ax_rf, color=["steelblue", "darkorange"], rot=0)
ax_rf.set_title("Recall por vocal: Random Forest base y ajustado")
ax_rf.set_xlabel("Vocal")
ax_rf.set_ylabel("Recall (%)")
ax_rf.set_ylim(0, 108)
ax_rf.set_yticks(np.arange(0, 101, 20))
ax_rf.legend(title="Modelo")
for contenedor_rf in ax_rf.containers:
    for barra_rf in contenedor_rf:
        ax_rf.text(barra_rf.get_x() + barra_rf.get_width() / 2, barra_rf.get_height() + 1,
                f"{barra_rf.get_height():.2f}", ha="center", va="bottom", fontsize=9)
ax_rf.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


Las barras comparan el recall sobre las mismas imágenes de prueba. Una barra más alta indica una mayor proporción de ejemplos de esa vocal reconocidos correctamente; no implica por sí sola una mejora equivalente en precision.


### Importancia de características

Cada importancia corresponde a un píxel de la representación de entrada preparada por el equipo. Se conserva la geometría de 100 × 75 mediante `reshape(ALTO, ANCHO)`. El índice plano de un píxel es `fila * ANCHO + columna`, con índices desde cero.


In [ ]:
importancias_rf = modelo_rf.feature_importances_


In [ ]:
importancias_rf_ajustado = mejor_modelo_rf.feature_importances_
assert (ALTO, ANCHO) == (100, 75)
assert len(importancias_rf) == len(importancias_rf_ajustado) == ALTO * ANCHO
limite_importancia_rf = max(importancias_rf.max(), importancias_rf_ajustado.max())
fig_rf, ejes_rf = plt.subplots(1, 2, figsize=(11, 7), constrained_layout=True)
for ax_rf, nombre_rf, valores_rf in zip(
    ejes_rf, ["Base", "Ajustado"], [importancias_rf, importancias_rf_ajustado]
):
    imagen_importancia_rf = ax_rf.imshow(valores_rf.reshape(ALTO, ANCHO), cmap="inferno",
                                     vmin=0, vmax=limite_importancia_rf)
    ax_rf.set_title(f"Importancia de píxeles — {nombre_rf}")
    ax_rf.set_xlabel("Columna")
    ax_rf.set_ylabel("Fila")
    ax_rf.set_xticks(np.linspace(0, ANCHO - 1, 4, dtype=int))
    ax_rf.set_yticks(np.linspace(0, ALTO - 1, 5, dtype=int))
    indice_pixel_rf = int(np.argmax(valores_rf))
    fila_pixel_rf, columna_pixel_rf = divmod(indice_pixel_rf, ANCHO)
    print(f"{nombre_rf}: píxel de mayor importancia = {indice_pixel_rf}, "
          f"fila = {fila_pixel_rf}, columna = {columna_pixel_rf}")
fig_rf.colorbar(imagen_importancia_rf, ax=ejes_rf, label="Importancia")
plt.show()


Los mapas sitúan los píxeles de mayor importancia en la geometría de entrada y comparten la misma escala de color. La intensidad resume la importancia asignada por cada bosque a sus divisiones; no demuestra causalidad ni implica que eliminar esos píxeles mantenga el desempeño.


### Comparación base vs. ajustado

La tabla y la gráfica se calculan con los modelos y las variables de esta ejecución, sobre las mismas particiones. La tabla expresa accuracy entre 0 y 1 y las barras lo muestran como porcentaje.


In [ ]:
comparacion_rf = pd.DataFrame({
    "Modelo": ["Base RF-1", "Ajustado RF-2"],
    "Accuracy de entrenamiento": [
        modelo_rf.score(X_train, y_train),
        accuracy_train_rf_ajustado
    ],
    "Accuracy de prueba": [
        modelo_rf.score(X_test, y_test),
        accuracy_test_rf_ajustado
    ]
})
comparacion_rf


In [ ]:
fig_rf, ax_rf = plt.subplots(figsize=(10, 5))
(comparacion_rf.set_index("Modelo") * 100).plot.bar(
    ax=ax_rf, color=["steelblue", "darkorange"], rot=0
)
ax_rf.set_title("Accuracy de Random Forest base y ajustado")
ax_rf.set_xlabel("Modelo")
ax_rf.set_ylabel("Accuracy (%)")
ax_rf.set_ylim(0, 108)
ax_rf.set_yticks(np.arange(0, 101, 20))
ax_rf.legend(title="Conjunto")
for contenedor_rf in ax_rf.containers:
    for barra_rf in contenedor_rf:
        ax_rf.text(barra_rf.get_x() + barra_rf.get_width() / 2, barra_rf.get_height() + 1,
                f"{barra_rf.get_height():.2f}", ha="center", va="bottom", fontsize=9)
ax_rf.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


La escala parte de cero para comparar las alturas sin exagerar las diferencias. La tabla y las barras describen el desempeño de ambos modelos sobre las mismas particiones; la selección del modelo ajustado procede de la validación cruzada de entrenamiento.


### Análisis y conclusiones del método

El resumen siguiente calcula la variación de accuracy y las confusiones entre O y U con los resultados de esta ejecución. Las tablas, figuras y conclusiones se obtienen directamente de los modelos y de las particiones comunes; no se reutilizan cifras de pruebas anteriores.


In [ ]:
cambio_accuracy_prueba_rf = (
    comparacion_rf.loc[1, "Accuracy de prueba"] - comparacion_rf.loc[0, "Accuracy de prueba"]
) * 100
print(f"Variación de accuracy de prueba (ajustado − base): {cambio_accuracy_prueba_rf:.4f} puntos porcentuales")
indice_o_rf, indice_u_rf = clases_rf.index("O"), clases_rf.index("U")
print(f"O → U: base = {matriz_confusion_rf[indice_o_rf, indice_u_rf]}, "
      f"ajustado = {matriz_confusion_rf_ajustado[indice_o_rf, indice_u_rf]}")
print(f"U → O: base = {matriz_confusion_rf[indice_u_rf, indice_o_rf]}, "
      f"ajustado = {matriz_confusion_rf_ajustado[indice_u_rf, indice_o_rf]}")


La variación de accuracy y las confusiones anteriores permiten valorar el cambio respecto al modelo base. La configuración elegida es la mejor según esta grilla y esta validación, sin demostrar que un criterio o una profundidad sean universalmente superiores. Las importancias describen el uso de los píxeles por el modelo, no relaciones causales.
